# Latch Stage 10 — Telegraph Evaluator Training

Train the `SENSOR_TELEMETRY_VERIFY` ModernBERT/LoRA evaluator on a Colab GPU using Telegraph's official evaluator kit and Latch's 64-pair corpus.

**Before running:** in Colab choose **Runtime → Change runtime type → T4 GPU** (or another NVIDIA GPU).

This notebook does **not** upload evaluation data or register the adapter on Telegraph. It only trains and packages local artifacts for the next validation/export step.

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. Select Runtime → Change runtime type → T4 GPU and reconnect.')

## 1. Clone Latch and Telegraph's official evaluator kit

In [ ]:
%cd /content
!rm -rf latch telegraph-wasm-baseline
!git clone --depth 1 https://github.com/Alike001/latch.git
!git clone --depth 1 https://github.com/telegraphprotocol/telegraph-wasm-baseline.git
!git -C latch log -1 --oneline
!git -C telegraph-wasm-baseline log -1 --oneline

## 2. Install evaluator dependencies without replacing Colab's CUDA-enabled PyTorch

In [ ]:
!python -m pip install -q --upgrade 'transformers>=4.48' peft numpy onnx onnxruntime datasets pyyaml safetensors
import torch, transformers, peft
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('peft:', peft.__version__)
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

## 3. Re-run the official dataset gate as a reproducibility check

In [ ]:
%cd /content/telegraph-wasm-baseline
!python -m float_runtime.train_adapter \
  --intent SENSOR_TELEMETRY_VERIFY \
  --data /content/latch/evaluator/sensor-telemetry/pairs.jsonl \
  --out /content/latch-artifacts/SENSOR_TELEMETRY_VERIFY \
  --config /content/latch/evaluator/sensor-telemetry/train-config.json \
  --check-data

The gate must report `PASS (64 pairs)` before training. If it fails, stop and preserve the output rather than using `--skip-dataset-gate`.

## 4. Train the real ModernBERT/LoRA evaluator

In [ ]:
%cd /content/telegraph-wasm-baseline
!mkdir -p /content/latch-artifacts/SENSOR_TELEMETRY_VERIFY
!python -m float_runtime.train_adapter \
  --intent SENSOR_TELEMETRY_VERIFY \
  --data /content/latch/evaluator/sensor-telemetry/pairs.jsonl \
  --out /content/latch-artifacts/SENSOR_TELEMETRY_VERIFY \
  --config /content/latch/evaluator/sensor-telemetry/train-config.json

Training is only accepted if `result.json` reports `verdict: PASS`. The output directory should contain `result.json`, an `adapter/` directory, and the reward head. On evaluator-kit commit `1ed063e`, a successful run may end with `NameError: pkg is not defined` while printing the suggested next command; this happens after the PASS result and artifacts are written. The safetensors delta is produced by the later export step.

## 5. Inspect and archive the training result

In [ ]:
from pathlib import Path
import json
out = Path('/content/latch-artifacts/SENSOR_TELEMETRY_VERIFY')
print('\n'.join(str(p.relative_to(out)) for p in sorted(out.rglob('*')) if p.is_file()))
result = json.loads((out / 'result.json').read_text())
summary = {k: result.get(k) for k in ['intent','train_pairs','gate_split','n_eval','pairwise_accuracy','val_loss','length_baseline','beats_length_baseline','verdict','fail_reason','negative_memorisation']}
print(json.dumps(summary, indent=2))
assert result.get('verdict') == 'PASS'

In [ ]:
!cd /content/latch-artifacts && zip -qr SENSOR_TELEMETRY_VERIFY-training.zip SENSOR_TELEMETRY_VERIFY
from google.colab import files
files.download('/content/latch-artifacts/SENSOR_TELEMETRY_VERIFY-training.zip')

## Next step after a training PASS

Bring the resulting training ZIP back to the Latch workflow. We will then:

1. obtain the released Telegraph Foundation metadata;
2. run `float_runtime.export_adapter`;
3. validate/package the LoRA delta with `cmd/adapterbuild`;
4. test the scorer locally;
5. check the live Evaluation Pool snapshot again before any registration claim.

Do **not** register a failed or unvalidated adapter.